# Frozen Lake - Value Iteration Implementation

This notebook implements a value iteration solution for the Frozen Lake environment.

In [5]:
import numpy as np
import gymnasium as gym

# Create the environment and reset it
env = gym.make('FrozenLake-v1', is_slippery=False)
state, info = env.reset(seed=42)
print('Initial state:', state)
print('Observation space:', env.observation_space)
print('Action space:', env.action_space)

n_states = env.observation_space.n
n_actions = env.action_space.n

# Value iteration
V = np.zeros(n_states)
gamma = 0.99
theta = 1e-8

for _ in range(100000):
    delta = 0
    for s in range(n_states):
        best_value = -np.inf
        for a in range(n_actions):
            q_value = 0.0
            for p, ns, r, terminated in env.unwrapped.P[s][a]:
                q_value += p * (r + gamma * V[ns])
            best_value = max(best_value, q_value)

        new_value = best_value
        delta = max(delta, abs(V[s] - new_value))
        V[s] = new_value

    if delta < theta:
        break

print('\nFinal value function:')
print(np.round(V.reshape(4, 4), 3))

# Extract greedy policy from the learned values
policy = np.zeros(n_states, dtype=int)
for s in range(n_states):
    best_action = 0
    best_q = -np.inf
    for a in range(n_actions):
        q_value = 0.0
        for p, ns, r, terminated in env.unwrapped.P[s][a]:
            q_value += p * (r + gamma * V[ns])
        if q_value > best_q:
            best_q = q_value
            best_action = a
    policy[s] = best_action

print('\nOptimal policy mapping:')
print(policy.reshape(4, 4))

# Demonstrate a short run using the learned policy
state, _ = env.reset(seed=42)
done = False
steps = 0
while not done and steps < 10:
    action = policy[state]
    state, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    steps += 1
    print(f'Step {steps}: action={action}, state={state}, reward={reward}, done={done}')


Initial state: 0
Observation space: Discrete(16)
Action space: Discrete(4)

Final value function:
[[0.951 0.961 0.97  0.961]
 [0.961 0.    0.98  0.   ]
 [0.97  0.98  0.99  0.   ]
 [0.    0.99  1.    0.   ]]

Optimal policy mapping:
[[1 2 1 0]
 [1 0 1 0]
 [2 1 1 0]
 [0 2 2 0]]
Step 1: action=1, state=4, reward=0, done=False
Step 2: action=1, state=8, reward=0, done=False
Step 3: action=2, state=9, reward=0, done=False
Step 4: action=1, state=13, reward=0, done=False
Step 5: action=2, state=14, reward=0, done=False
Step 6: action=2, state=15, reward=1, done=True
